In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os, sys
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"
src_path = "/content/drive/MyDrive/llm_from_scratch/src"
sys.path.append(data_path)
sys.path.append(src_path)

In [8]:
os.chdir(data_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/datasets


In [11]:
import json
with open('instruction_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [12]:
data[50]

{'instruction': 'Identify the correct spelling of the following word.',
 'input': 'Ocassion',
 'output': "The correct spelling is 'Occasion.'"}

In [13]:
def format_input(text):
    instruction_text = (
        f"Below is an instruction that defines a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{text['instruction']}"
    )
    input_text = f"\n\n### Input:\n{text['input']}" if text['input'] else ""
    return instruction_text + input_text

In [33]:
print(len(data))
print(format_input(data[50]))

1100
Below is an instruction that defines a task.Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion


In [41]:
# partitioning dataset into train, validation and test:
train_part = int(len(data) * 0.85)
test_part = int(len(data) * 0.1)
val_part = int(len(data) - train_part - test_part)

In [42]:
train_data = data[:train_part]
test_data = data[train_part:train_part + test_part]
val_data = data[train_part + test_part:]

In [43]:
print(len(train_data))
print(len(test_data))
print(len(val_data))

935
110
55


In [48]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [49]:
from torch.utils.data import Dataset, DataLoader
import torch
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_text = []

        for text in self.data:
            instruction_plus_input = format_input(text)
            response_text = f"\n\n### Response:\n{text['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_text.append(tokenizer.encode(full_text)

            )

    def __len__(self):
        return len(self.encoded_text)

    def __getitem__(self, idx):
        return torch.tensor(self.encoded_text[idx])

In [50]:
# function to create each input to equal length in a batch:
def collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device='cpu'):
    batch_max_length = max([len(item)+1 for item in batch])
    input_lst, target_lst = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).unsqueeze()

        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        input_lst.append(inputs).to(device)
        target_lst.append(targets).to(device)

In [52]:
num_workers = 0
batch_size = 8

torch.manual_seed(123)


train_dataset = InstructionDataset(train_data, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
    )
val_dataset = InstructionDataset(val_data, tokenizer)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )
test_dataset = InstructionDataset(test_data, tokenizer)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

In [56]:
print(len(train_loader),
len(test_loader),
len(val_loader))

116 14 7
